# 02 Run Models

This notebook loads the prepared parquet/CSV artifacts from `outputs/prepared` and runs the forecasting models, evaluation tables, and plots.


In [ ]:
# Optional for Colab / fresh environments:
# !pip install openassetpricing pyarrow statsmodels scikit-learn openpyxl seaborn


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
from scipy.linalg import qr
from scipy.stats import norm
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNet, Ridge, enet_path

sns.set_theme(style="whitegrid", context="talk")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=ConvergenceWarning)

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PREPARED_DIR = OUTPUT_DIR / "prepared"
FORECAST_DIR = OUTPUT_DIR / "forecasts"
TABLE_DIR = OUTPUT_DIR / "tables"
for folder in [OUTPUT_DIR, PREPARED_DIR, FORECAST_DIR, TABLE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

SAVE_FORECASTS = True
LOAD_SAVED_FORECASTS = False

SAMPLE_START = pd.Period("1970-01", freq="M")
FORECAST_START = pd.Period("1985-01", freq="M")
TRAINING_WINDOW = 120
VALIDATION_WINDOW = 60
ROLLING_ESTIMATION_WINDOW = TRAINING_WINDOW + VALIDATION_WINDOW
MARKET_VAR_WINDOW = 60
GAMMA = 3.0
ENET_MAX_ITER = 100000
ENET_TOL = 1e-6
ENET_LAMBDA_GRID = np.logspace(-4, 1, 50)
RIDGE_LAMBDA_GRID = ENET_LAMBDA_GRID.copy()


In [ ]:
gw_model = pd.read_parquet(PREPARED_DIR / "gw_model.parquet")
ls_model = pd.read_parquet(PREPARED_DIR / "ls_model.parquet")
target_panel = pd.read_parquet(PREPARED_DIR / "target_panel.parquet")
gw_features = pd.read_parquet(PREPARED_DIR / "gw_features.parquet")
ls_features = pd.read_parquet(PREPARED_DIR / "ls_features.parquet")
ls_metadata = pd.read_parquet(PREPARED_DIR / "ls_metadata.parquet")

for df in [gw_model, ls_model, target_panel, gw_features, ls_features, ls_metadata]:
    if isinstance(df.index, pd.Index):
        try:
            df.index = pd.PeriodIndex(df.index, freq="M")
        except Exception:
            pass

print(f"Loaded prepared data from: {PREPARED_DIR}")
print(f"GW model sample: {gw_model.index.min()} to {gw_model.index.max()} ({len(gw_model)} rows)")
print(f"LS model sample: {ls_model.index.min()} to {ls_model.index.max()} ({len(ls_model)} rows)")


In [ ]:
def ols_forecast_scalar(x_train: np.ndarray, y_train: np.ndarray, x_new: float) -> float:
    mask = np.isfinite(x_train) & np.isfinite(y_train)
    x = np.asarray(x_train[mask], dtype=float)
    y = np.asarray(y_train[mask], dtype=float)
    if len(y) == 0:
        return np.nan
    if len(y) < 3 or np.isclose(np.nanstd(x), 0.0):
        return float(np.nanmean(y))
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return float(np.array([1.0, x_new]) @ beta)


def ols_predict_vector(x_train: np.ndarray, y_train: np.ndarray, x_pred: np.ndarray) -> np.ndarray:
    mask = np.isfinite(x_train) & np.isfinite(y_train)
    x = np.asarray(x_train[mask], dtype=float)
    y = np.asarray(y_train[mask], dtype=float)
    x_pred = np.asarray(x_pred, dtype=float)
    if len(y) == 0:
        return np.full(len(x_pred), np.nan)
    if len(y) < 3 or np.isclose(np.nanstd(x), 0.0):
        return np.full(len(x_pred), float(np.nanmean(y)))
    X = np.column_stack([np.ones(len(x)), x])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return np.column_stack([np.ones(len(x_pred)), x_pred]) @ beta


def select_ols_independent_columns(X: np.ndarray) -> np.ndarray:
    if X.ndim == 1:
        return np.array([True])
    design = np.column_stack([np.ones(len(X)), X])
    _, r, piv = qr(design, mode="economic", pivoting=True)
    diag = np.abs(np.diag(r))
    if diag.size == 0:
        return np.zeros(X.shape[1], dtype=bool)
    tol = np.finfo(float).eps * max(design.shape) * diag.max()
    rank = int((diag > tol).sum())
    keep = np.zeros(X.shape[1], dtype=bool)
    for idx in piv[:rank]:
        if idx > 0:
            keep[idx - 1] = True
    return keep


def multivariate_ols_forecast(X_train: np.ndarray, y_train: np.ndarray, x_new: np.ndarray) -> float:
    mask = np.isfinite(y_train) & np.isfinite(X_train).all(axis=1)
    X = np.asarray(X_train[mask], dtype=float)
    y = np.asarray(y_train[mask], dtype=float)
    x_new = np.asarray(x_new, dtype=float)
    if len(y) == 0:
        return np.nan
    if X.ndim == 1:
        return ols_forecast_scalar(X, y, float(x_new))
    active = np.nanstd(X, axis=0) > 1e-12
    if not active.any():
        return float(np.nanmean(y))
    X = X[:, active]
    x_new = x_new[active]
    independent = select_ols_independent_columns(X)
    if not independent.any():
        return float(np.nanmean(y))
    X = X[:, independent]
    x_new = x_new[independent]
    design = np.column_stack([np.ones(len(X)), X])
    beta = np.linalg.lstsq(design, y, rcond=None)[0]
    return float(np.r_[1.0, x_new] @ beta)


def fit_enet_with_lambda(X: np.ndarray, y: np.ndarray, lambda_value: float, positive: bool = False) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X = X[mask]
    y = y[mask]
    p = X.shape[1]
    if len(y) == 0:
        return {"intercept": np.nan, "coef": np.full(p, np.nan), "lambda": lambda_value}

    active = np.nanstd(X, axis=0) > 1e-12
    coef_full = np.zeros(p)
    if not active.any():
        return {"intercept": float(np.mean(y)), "coef": coef_full, "lambda": lambda_value}

    model = ElasticNet(
        alpha=lambda_value,
        l1_ratio=0.5,
        fit_intercept=True,
        max_iter=ENET_MAX_ITER,
        tol=ENET_TOL,
        positive=positive,
    )
    model.fit(X[:, active], y)
    coef_full[active] = model.coef_
    return {"intercept": float(model.intercept_), "coef": coef_full, "lambda": float(lambda_value)}


def fit_ridge_with_lambda(X: np.ndarray, y: np.ndarray, lambda_value: float) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X = X[mask]
    y = y[mask]
    p = X.shape[1]
    if len(y) == 0:
        return {"intercept": np.nan, "coef": np.full(p, np.nan), "lambda": lambda_value}

    active = np.nanstd(X, axis=0) > 1e-12
    coef_full = np.zeros(p)
    if not active.any():
        return {"intercept": float(np.mean(y)), "coef": coef_full, "lambda": lambda_value}

    model = Ridge(alpha=lambda_value, fit_intercept=True)
    model.fit(X[:, active], y)
    coef_full[active] = model.coef_
    return {"intercept": float(model.intercept_), "coef": coef_full, "lambda": float(lambda_value)}


def estimate_enet_validation(X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray, positive: bool = False) -> dict:
    best_lambda = np.nan
    best_mse = np.inf
    for lambda_value in ENET_LAMBDA_GRID:
        fit = fit_enet_with_lambda(X_train, y_train, lambda_value, positive=positive)
        if np.isnan(fit["intercept"]):
            continue
        val_pred = fit["intercept"] + np.asarray(X_val, dtype=float) @ fit["coef"]
        mask = np.isfinite(y_val) & np.isfinite(val_pred)
        if not mask.any():
            continue
        mse = float(np.mean((np.asarray(y_val, dtype=float)[mask] - val_pred[mask]) ** 2))
        if mse < best_mse:
            best_mse = mse
            best_lambda = float(lambda_value)

    return {"lambda": best_lambda, "val_mse": best_mse}


def estimate_ridge_validation(X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray) -> dict:
    best_lambda = np.nan
    best_mse = np.inf
    for lambda_value in RIDGE_LAMBDA_GRID:
        fit = fit_ridge_with_lambda(X_train, y_train, lambda_value)
        if np.isnan(fit["intercept"]):
            continue
        val_pred = fit["intercept"] + np.asarray(X_val, dtype=float) @ fit["coef"]
        mask = np.isfinite(y_val) & np.isfinite(val_pred)
        if not mask.any():
            continue
        mse = float(np.mean((np.asarray(y_val, dtype=float)[mask] - val_pred[mask]) ** 2))
        if mse < best_mse:
            best_mse = mse
            best_lambda = float(lambda_value)

    return {"lambda": best_lambda, "val_mse": best_mse}


def estimate_enet_aicc(X: np.ndarray, y: np.ndarray, positive: bool = False) -> dict:
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    mask = np.isfinite(y) & np.isfinite(X).all(axis=1)
    X = X[mask]
    y = y[mask]
    p = X.shape[1]
    if len(y) == 0:
        return {"intercept": np.nan, "coef": np.full(p, np.nan), "lambda": np.nan, "r2": np.nan}

    active = np.nanstd(X, axis=0) > 1e-12
    coef_full = np.zeros(p)
    if not active.any():
        return {"intercept": float(np.mean(y)), "coef": coef_full, "lambda": np.nan, "r2": 0.0}

    X_work = X[:, active]
    x_mean = X_work.mean(axis=0)
    y_mean = y.mean()
    X_centered = X_work - x_mean
    y_centered = y - y_mean

    alphas, coefs, _ = enet_path(
        X_centered,
        y_centered,
        l1_ratio=0.5,
        n_alphas=200,
        positive=positive,
        max_iter=ENET_MAX_ITER,
        tol=ENET_TOL,
    )
    intercepts = y_mean - x_mean @ coefs
    preds = intercepts + X_work @ coefs
    rss = ((y[:, None] - preds) ** 2).sum(axis=0)
    tss = ((y - y_mean) ** 2).sum()
    df = (np.abs(coefs) > 1e-10).sum(axis=0)
    denom = len(y) - df - 1
    aicc = np.full_like(rss, np.inf, dtype=float)
    valid = denom > 0
    aicc[valid] = len(y) * np.log(rss[valid] / len(y)) + 2 * df[valid] * len(y) / denom[valid]
    best = int(np.argmin(aicc))

    coef_full[active] = coefs[:, best]
    r2 = np.nan if np.isclose(tss, 0.0) else float(1 - rss[best] / tss)
    return {
        "intercept": float(intercepts[best]),
        "coef": coef_full,
        "lambda": float(alphas[best]),
        "r2": r2,
    }


def cw_test(actual: pd.Series, benchmark: pd.Series, forecast: pd.Series, hac_lags: int = 1) -> dict:
    joined = pd.concat([actual, benchmark, forecast], axis=1).dropna()
    joined.columns = ["actual", "benchmark", "forecast"]
    e_bench = joined["actual"] - joined["benchmark"]
    e_model = joined["actual"] - joined["forecast"]
    msfe_bench = np.mean(e_bench**2)
    msfe_model = np.mean(e_model**2)
    r2_oos = 100 * (1 - msfe_model / msfe_bench)
    f = e_bench**2 - e_model**2 + (joined["benchmark"] - joined["forecast"]) ** 2
    fit = sm.OLS(f.values, np.ones((len(f), 1))).fit(cov_type="HAC", cov_kwds={"maxlags": hac_lags})
    stat = float(fit.params[0] / fit.bse[0])
    p_value = float(1 - norm.cdf(stat))
    return {"r2_oos": float(r2_oos), "cw_stat": stat, "cw_pvalue": p_value}


def timing_metrics(actual_excess: pd.Series, rf: pd.Series, forecast: pd.Series, benchmark: pd.Series) -> dict:
    market_var = actual_excess.shift(1).rolling(MARKET_VAR_WINDOW).var()
    joined = pd.concat([actual_excess, rf, forecast, benchmark, market_var], axis=1).dropna()
    joined.columns = ["actual_excess", "rf", "forecast", "benchmark", "market_var"]

    def portfolio_stats(signal: pd.Series) -> tuple[float, float, float, float]:
        weight = (1 / GAMMA) * (signal / joined["market_var"])
        weight = weight.clip(-1, 2)
        rp = weight * joined["actual_excess"] + joined["rf"]
        ann_ret = float(rp.mean() * 12 * 100)
        ann_vol = float(rp.std(ddof=0) * np.sqrt(12) * 100)
        sharpe = np.nan if np.isclose(ann_vol, 0.0) else ann_ret / ann_vol
        cer = float((rp.mean() - 0.5 * GAMMA * rp.var(ddof=0)) * 12 * 100)
        return ann_ret, ann_vol, sharpe, cer

    ann_ret, ann_vol, sharpe, cer = portfolio_stats(joined["forecast"])
    _, _, _, cer_bench = portfolio_stats(joined["benchmark"])
    return {
        "ann_return": ann_ret,
        "ann_vol": ann_vol,
        "sharpe": sharpe,
        "cer_delta": cer - cer_bench,
    }


def save_wide(df: pd.DataFrame, stem: str) -> None:
    if not SAVE_FORECASTS:
        return
    csv_path = FORECAST_DIR / f"{stem}.csv"
    parquet_path = FORECAST_DIR / f"{stem}.parquet"
    df.to_csv(csv_path)
    df.to_parquet(parquet_path)


def save_table(df: pd.DataFrame, stem: str) -> None:
    csv_path = TABLE_DIR / f"{stem}.csv"
    parquet_path = TABLE_DIR / f"{stem}.parquet"
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)


def make_scheme_indices(pos: int, scheme: str) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    val_start = pos - VALIDATION_WINDOW
    if scheme == "expanding":
        train_idx = np.arange(0, val_start)
        full_idx = np.arange(0, pos)
    elif scheme == "rolling":
        train_idx = np.arange(val_start - TRAINING_WINDOW, val_start)
        full_idx = np.arange(pos - ROLLING_ESTIMATION_WINDOW, pos)
    else:
        raise ValueError(f"Unknown scheme: {scheme}")

    val_idx = np.arange(val_start, pos)
    return train_idx, val_idx, full_idx


def run_forecast_scheme(model_df: pd.DataFrame, predictor_set: str, include_avg: bool = False, scheme: str = "expanding") -> dict:
    predictors = [col for col in model_df.columns if col not in {"mkt_excess", "Rfree"}]
    y = model_df["mkt_excess"].to_numpy(dtype=float)
    rf = model_df["Rfree"].to_numpy(dtype=float)
    X = model_df[predictors].to_numpy(dtype=float)
    avg_signal = model_df[predictors].mean(axis=1).to_numpy(dtype=float) if include_avg else None

    eval_index = model_df.index[model_df.index >= FORECAST_START]
    eval_positions = [model_df.index.get_loc(month) for month in eval_index]

    uni_forecasts = pd.DataFrame(index=eval_index, columns=predictors, dtype=float)
    model_names = ["Ols", "Enet", "EnetVal", "RidgeVal", "Combine", "Cenet"] + (["Avg"] if include_avg else [])
    model_forecasts = pd.DataFrame(index=eval_index, columns=model_names, dtype=float)
    benchmark = pd.Series(index=eval_index, dtype=float, name="PrevailingMean")
    actual = pd.Series(index=eval_index, dtype=float, name="Actual")
    rf_eval = pd.Series(index=eval_index, dtype=float, name="Rfree")

    for step, pos in enumerate(eval_positions, start=1):
        month = model_df.index[pos]
        train_idx, val_idx, full_idx = make_scheme_indices(pos, scheme)
        actual.loc[month] = y[pos]
        rf_eval.loc[month] = rf[pos]
        benchmark.loc[month] = float(np.mean(y[full_idx]))

        current_uni = np.empty(len(predictors), dtype=float)
        for j, _ in enumerate(predictors):
            current_uni[j] = ols_forecast_scalar(X[full_idx, j], y[full_idx], X[pos, j])
        uni_forecasts.loc[month] = current_uni

        model_forecasts.loc[month, "Combine"] = float(np.nanmean(current_uni))
        model_forecasts.loc[month, "Ols"] = multivariate_ols_forecast(X[full_idx], y[full_idx], X[pos])

        enet_direct = estimate_enet_aicc(X[full_idx], y[full_idx], positive=False)
        model_forecasts.loc[month, "Enet"] = float(enet_direct["intercept"] + X[pos] @ enet_direct["coef"])

        enet_val = estimate_enet_validation(X[train_idx], y[train_idx], X[val_idx], y[val_idx], positive=False)
        enet_val_refit = fit_enet_with_lambda(X[full_idx], y[full_idx], enet_val["lambda"], positive=False)
        model_forecasts.loc[month, "EnetVal"] = float(enet_val_refit["intercept"] + X[pos] @ enet_val_refit["coef"])

        ridge_val = estimate_ridge_validation(X[train_idx], y[train_idx], X[val_idx], y[val_idx])
        ridge_val_refit = fit_ridge_with_lambda(X[full_idx], y[full_idx], ridge_val["lambda"])
        model_forecasts.loc[month, "RidgeVal"] = float(ridge_val_refit["intercept"] + X[pos] @ ridge_val_refit["coef"])

        val_uni = np.column_stack(
            [
                ols_predict_vector(X[train_idx, j], y[train_idx], X[val_idx, j])
                for j in range(len(predictors))
            ]
        )
        cenet = estimate_enet_aicc(val_uni, y[val_idx], positive=True)
        selected = np.flatnonzero(np.abs(cenet["coef"]) > 1e-10)
        if len(selected) == 0:
            model_forecasts.loc[month, "Cenet"] = benchmark.loc[month]
        else:
            model_forecasts.loc[month, "Cenet"] = float(np.nanmean(current_uni[selected]))

        if include_avg:
            model_forecasts.loc[month, "Avg"] = ols_forecast_scalar(
                avg_signal[full_idx],
                y[full_idx],
                avg_signal[pos],
            )

        if step % 24 == 0:
            print(f"{predictor_set} | {scheme} | processed {step}/{len(eval_positions)} forecast months")

    outputs = {
        "actual": actual.to_frame(),
        "rf": rf_eval.to_frame(),
        "benchmark": benchmark.to_frame(),
        "univariate": uni_forecasts,
        "models": model_forecasts,
    }
    save_wide(outputs["actual"], f"actual_{scheme}")
    save_wide(outputs["rf"], f"rf_{scheme}")
    save_wide(outputs["benchmark"], f"benchmark_{scheme}")
    save_wide(outputs["univariate"], f"univariate_{predictor_set}_{scheme}")
    save_wide(outputs["models"], f"models_{predictor_set}_{scheme}")
    return outputs


def summarize_matrix(actual: pd.Series, rf: pd.Series, benchmark: pd.Series, matrix: pd.DataFrame, predictor_set: str) -> pd.DataFrame:
    rows = []
    for col in matrix.columns:
        stats = cw_test(actual, benchmark, matrix[col], hac_lags=1)
        econ = timing_metrics(actual, rf, matrix[col], benchmark)
        rows.append(
            {
                "name": col,
                "predictor_set": predictor_set,
                **stats,
                **econ,
            }
        )
    return pd.DataFrame(rows).sort_values("r2_oos", ascending=False).reset_index(drop=True)


In [ ]:
if LOAD_SAVED_FORECASTS:
    results = {}
    for scheme in ["expanding", "rolling"]:
        results[scheme] = {}
        for predictor_set in ["gw", "ls"]:
            results[scheme][predictor_set] = {
                "actual": pd.read_parquet(FORECAST_DIR / f"actual_{scheme}.parquet"),
                "rf": pd.read_parquet(FORECAST_DIR / f"rf_{scheme}.parquet"),
                "benchmark": pd.read_parquet(FORECAST_DIR / f"benchmark_{scheme}.parquet"),
                "univariate": pd.read_parquet(FORECAST_DIR / f"univariate_{predictor_set}_{scheme}.parquet"),
                "models": pd.read_parquet(FORECAST_DIR / f"models_{predictor_set}_{scheme}.parquet"),
            }
else:
    results = {
        "expanding": {
            "gw": run_forecast_scheme(gw_model, "gw", include_avg=False, scheme="expanding"),
            "ls": run_forecast_scheme(ls_model, "ls", include_avg=True, scheme="expanding"),
        },
        "rolling": {
            "gw": run_forecast_scheme(gw_model, "gw", include_avg=False, scheme="rolling"),
            "ls": run_forecast_scheme(ls_model, "ls", include_avg=True, scheme="rolling"),
        },
    }


In [ ]:
summary_tables = {}

for scheme in ["expanding", "rolling"]:
    summary_tables[scheme] = {}
    for predictor_set in ["gw", "ls"]:
        actual = results[scheme][predictor_set]["actual"]["Actual"]
        rf_eval = results[scheme][predictor_set]["rf"]["Rfree"]
        benchmark = results[scheme][predictor_set]["benchmark"]["PrevailingMean"]

        uni_summary = summarize_matrix(
            actual,
            rf_eval,
            benchmark,
            results[scheme][predictor_set]["univariate"],
            predictor_set.upper(),
        )
        model_summary = summarize_matrix(
            actual,
            rf_eval,
            benchmark,
            results[scheme][predictor_set]["models"],
            predictor_set.upper(),
        )

        summary_tables[scheme][predictor_set] = {
            "univariate": uni_summary,
            "models": model_summary,
        }

        save_table(uni_summary, f"table_univariate_{predictor_set}_{scheme}")
        save_table(model_summary, f"table_models_{predictor_set}_{scheme}")

display(summary_tables["expanding"]["gw"]["models"].head(10))
display(summary_tables["expanding"]["ls"]["models"].head(10))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 12), sharex=False, sharey=False)
plot_specs = [
    ("rolling", "gw", axes[0, 0], "GW Univariate R^2_OOS | Rolling"),
    ("rolling", "ls", axes[0, 1], "LS Univariate R^2_OOS | Rolling"),
    ("expanding", "gw", axes[1, 0], "GW Univariate R^2_OOS | Expanding"),
    ("expanding", "ls", axes[1, 1], "LS Univariate R^2_OOS | Expanding"),
]

for scheme, predictor_set, ax, title in plot_specs:
    data = summary_tables[scheme][predictor_set]["univariate"]["r2_oos"]
    sns.histplot(data, bins=25, kde=False, ax=ax, color="#2f5d50")
    ax.axvline(data.mean(), color="#b03a2e", linestyle="--", linewidth=2, label=f"Mean = {data.mean():.2f}")
    ax.set_title(title)
    ax.set_xlabel("R^2_OOS")
    ax.legend()

plt.tight_layout()
plt.show()


Run `01_prepare_data.ipynb` first so the prepared parquet/CSV files exist.
